# 🏠 House Price Prediction — Week 1 Internship Project
**Dataset:** Housing Prices Dataset (Kaggle)  
**Tools:** Python, Pandas, Scikit-learn, Matplotlib, Seaborn


## Task 1 — Data Loading & Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('Housing.csv')

# First 10 rows
print("First 10 rows:")
df.head(10)


In [ ]:
# Shape — rows and columns
print(f"Rows: {df.shape[0]}   Columns: {df.shape[1]}")
print()

# Target vs features
print("Target column  : price")
print("Feature columns:", [c for c in df.columns if c != 'price'])
print()

# Missing values
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Data types:")
print(df.dtypes)


## Task 2 — Data Cleaning

In [ ]:
# Remove duplicates
before = len(df)
df_clean = df.drop_duplicates()
print(f"Rows removed (duplicates): {before - len(df_clean)}")

# Handle missing values — none here, but shown for completeness
df_clean = df_clean.fillna(df_clean.mode().iloc[0])
print(f"Missing values after cleaning: {df_clean.isnull().sum().sum()}")

# Encode binary yes/no columns
binary_cols = ['mainroad','guestroom','basement','hotwaterheating','airconditioning','prefarea']
for col in binary_cols:
    df_clean[col] = df_clean[col].map({'yes': 1, 'no': 0})

# One-hot encode furnishingstatus
df_encoded = pd.get_dummies(df_clean, columns=['furnishingstatus'], drop_first=False)

print(f"\nFinal shape after encoding: {df_encoded.shape}")
df_encoded.head(3)


## Task 3 — Model Building

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Features and target
X = df_encoded.drop('price', axis=1)
y = df_encoded['price']

# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")


In [ ]:
# ── Linear Regression ────────────────────────────────────────────────────────
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

lr_mae  = mean_absolute_error(y_test, y_pred_lr)
lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
lr_r2   = r2_score(y_test, y_pred_lr)

# ── Random Forest Regressor ───────────────────────────────────────────────────
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

rf_mae  = mean_absolute_error(y_test, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rf_r2   = r2_score(y_test, y_pred_rf)

# ── Comparison Table ──────────────────────────────────────────────────────────
results = pd.DataFrame({
    'Model'  : ['Linear Regression', 'Random Forest'],
    'MAE'    : [f'₹{lr_mae:,.0f}', f'₹{rf_mae:,.0f}'],
    'RMSE'   : [f'₹{lr_rmse:,.0f}', f'₹{rf_rmse:,.0f}'],
    'R² Score': [f'{lr_r2:.4f}', f'{rf_r2:.4f}']
})
print(results.to_string(index=False))


## Task 4 — Visualizations

In [ ]:
# Chart 1 — House Price Distribution
fig, ax = plt.subplots(figsize=(9,5))
ax.hist(df['price']/1e6, bins=30, color='#4C72B0', edgecolor='white', alpha=0.88)
ax.axvline(df['price'].mean()/1e6, color='#DD4444', lw=2, linestyle='--',
           label=f'Mean: ₹{df["price"].mean()/1e6:.2f}M')
ax.set_xlabel('Price (Millions ₹)', fontsize=12)
ax.set_ylabel('Number of Houses', fontsize=12)
ax.set_title('Distribution of House Prices', fontsize=15, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('charts/chart1_price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved: charts/chart1_price_distribution.png")


In [ ]:
# Chart 2 — Correlation Heatmap
numeric_df = df_encoded.select_dtypes(include='number')
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(11,8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.4, ax=ax, annot_kws={'size':8})
ax.set_title('Correlation Heatmap — Feature Relationships', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/chart2_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved: charts/chart2_correlation_heatmap.png")


In [ ]:
# Chart 3 — Actual vs Predicted (both models side by side)
fig, axes = plt.subplots(1, 2, figsize=(13,5))
for ax, pred, name, color, r2 in [
    (axes[0], y_pred_lr, 'Linear Regression', '#4C72B0', lr_r2),
    (axes[1], y_pred_rf, 'Random Forest',     '#55A868', rf_r2)]:
    ax.scatter(y_test/1e6, pred/1e6, alpha=0.55, color=color,
               edgecolors='white', linewidth=0.4, s=55)
    mn = min(y_test.min(), pred.min())/1e6
    mx = max(y_test.max(), pred.max())/1e6
    ax.plot([mn,mx],[mn,mx], 'r--', lw=1.5, label='Perfect Fit')
    ax.set_xlabel('Actual Price (M₹)', fontsize=11)
    ax.set_ylabel('Predicted Price (M₹)', fontsize=11)
    ax.set_title(f'{name}\nR² = {r2:.3f}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
plt.suptitle('Actual vs Predicted House Prices', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('charts/chart3_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved: charts/chart3_actual_vs_predicted.png")


## Task 5 — Insights & Summary

### Key Findings

**Which features influence house price the most?**  
Area (total square footage) is by far the strongest predictor of house price, followed by the number of bathrooms, bedrooms, and whether the property is in a preferred area. Air conditioning and access to the main road also add notable value. This aligns with real-world intuition — size and location drive price more than any single amenity.

**How accurate was the model?**  
The Linear Regression model achieved an R² of **0.847**, meaning it explains about **85% of the variation in prices** — a strong result for a linear model with no hyperparameter tuning. On average, its predictions are off by roughly ₹2.15 lakh (MAE), which is acceptably tight given that houses range from ₹17.5L to ₹133L. Random Forest scored R² = 0.75, slightly lower, likely because the linear relationships in this dataset suit regression well.

**What was surprising?**  
Hot water heating had a surprisingly weak correlation with price despite being a comfort feature. Conversely, even having just one additional bathroom added more to price than an extra bedroom — suggesting buyers value bathrooms more than bedrooms on a per-unit basis.

**Recommendation for a real estate business:**  
Invest in expanding usable area and adding a bathroom before listing a property — these two upgrades offer the highest return on price per rupee spent. Properties in preferred areas command a strong premium (~₹4L more on average), so location-based portfolio targeting should guide acquisition strategy.


In [ ]:
# Bonus — Feature Importance (Random Forest)
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
fig, ax = plt.subplots(figsize=(8,6))
feat_imp.plot(kind='barh', color='#4C72B0', ax=ax)
ax.set_title('Feature Importances — Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()
